# V73 - HGB grouped confirm

Pre-registered plan: an ungrouped holdout once showed 0.63362 for HistGradientBoosting, but ungrouped numbers are inflated by template leakage, so confirm under the standard grouped protocol with a fresh seed. Same word plus assessment features for both contenders, so only the learner differs. Gate with the model-class floor and worst-fold lock.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 515
N_SPLITS = 5
GATE_FLOOR = 0.003
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v73")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Grouped HGB vs Ridge, identical features

HGB runs 300 iterations per module per fold. Slow but decisive.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
X = np.hstack([W, A]).astype(np.float32)
X_tr = X[:n_train]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_r, oof_h, fids = np.zeros_like(Y), np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(X_tr, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(X_tr[tr], Y[tr, j])
        oof_r[va, j] = m.predict(X_tr[va])
        h = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, early_stopping=False, random_state=SEED)
        h.fit(X_tr[tr], Y[tr, j])
        oof_h[va, j] = h.predict(X_tr[va])
    fids[va] = fi + 1
    print(f"fold {fi + 1}: ridge={ndcg5(Y[va], oof_r[va]).mean():.5f} hgb={ndcg5(Y[va], oof_h[va]).mean():.5f}", flush=True)
base = ndcg5(Y, oof_r)
d = ndcg5(Y, oof_h) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_h[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"ridge={base.mean():.5f} hgb={(base + d).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT HGB" if (lo > GATE_FLOOR and wf >= 0) else "KEEP Ridge; HGB holdout was leakage-inflated or noise")

fold 1: ridge=0.63905 hgb=0.63545


fold 2: ridge=0.62939 hgb=0.60870


fold 3: ridge=0.63766 hgb=0.63107


fold 4: ridge=0.57526 hgb=0.61379


fold 5: ridge=0.62880 hgb=0.63197


ridge=0.62727 hgb=0.62653 delta=-0.00074 CI95=[-0.00505,+0.00405] worst-fold=-0.02069
KEEP Ridge; HGB holdout was leakage-inflated or noise
